# ViFinQA P2.2 — Stage-B OOM tail recovery

RETIRED: do not run. Use `vifinqa-codegen-p22.ipynb` with payload schema 7 and the B-groundable mask.

In [ ]:
raise RuntimeError('RETIRED notebook: use vifinqa-codegen-p22.ipynb schema 7')
import glob, json, pathlib
hits = glob.glob('/kaggle/input/**/retrieval.jsonl', recursive=True)
assert len(hits) == 1, f'Attach exactly one payload dataset: {hits}'
PAYLOAD = str(pathlib.Path(hits[0]).parent)
manifest = json.loads((pathlib.Path(PAYLOAD) / 'payload-manifest.json').read_text(encoding='utf-8'))
assert manifest.get('schema_version') == 6, manifest.get('schema_version')
assert manifest.get('fuzzy_scorer') == {'backend': 'difflib.SequenceMatcher', 'version': '1'}
MASK_REL = 'targets/p22b_oom_tail.json'
mask_path = pathlib.Path(PAYLOAD) / MASK_REL
mask = json.loads(mask_path.read_text(encoding='utf-8'))
assert mask.get('schema_version') == 'p22_oom_tail_mask_v1'
assert mask['count'] == len(mask['ids']) == len(set(mask['ids'])) > 0
assert MASK_REL in manifest['files']
print('PAYLOAD', PAYLOAD, '| schema', manifest['schema_version'], '| files', len(manifest['files']), '| tail', mask['count'], mask['ids'])

In [ ]:
import shutil
SRC = pathlib.Path(PAYLOAD) / 'code'
DST = pathlib.Path('/kaggle/working/code')
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print('code ->', DST)

In [ ]:
%%time
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"
import transformers, bitsandbytes, torch
print('transformers', transformers.__version__, 'bitsandbytes', bitsandbytes.__version__, 'GPUs', torch.cuda.device_count())

## Run only the frozen tail

Batch size and checkpoint size are one. If `n=2` exhausts memory, schema 6 retries the two samples sequentially as `n=1 + n=1`. The lower input/output limits and the new mask intentionally create a new run signature.

In [ ]:
%%time
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --load-4bit \
    --llm-mode select_v2 --llm-target empty \
    --llm-ids-file targets/p22b_oom_tail.json \
    --out /kaggle/working/codegen_p22b_oom_tail_sel14b.jsonl \
    --n 2 --temperature 0.2 --k 0 --max-tokens 512 --max-input-tokens 6000 \
    --batch-size 1 --checkpoint-every 1 --time-budget-min 120 --seed 13

In [ ]:
import collections, math
out = pathlib.Path('/kaggle/working/codegen_p22b_oom_tail_sel14b.jsonl')
rows = [json.loads(line) for line in out.open(encoding='utf-8')]
attempted = {int(row['id']) for row in rows if row.get('llm_attempt_status') == 'completed'}
target = set(map(int, mask['ids']))
assert len(rows) == 1012 and len({int(row['id']) for row in rows}) == 1012
assert all(math.isfinite(float(row['answer'])) for row in rows)
assert len({row.get('run_signature') for row in rows}) == 1
assert attempted == target, {'pending': sorted(target - attempted), 'outside': sorted(attempted - target)}
print('tail complete', len(attempted), 'outcomes', collections.Counter((row.get('selection_trace') or {}).get('outcome') for row in rows if int(row['id']) in attempted))

Download `/kaggle/working/codegen_p22b_oom_tail_sel14b.jsonl`. Audit and merge it locally with the saved partial checkpoint using the two-step commands in `RUNBOOK_P2_2_STRUCTURED_SELECTION_V2.md`. Do not run Stage C until the recovered Stage-B submission is built and saved.